# Reading another project's results

`crossrepo` lets one project read result files produced by another, without
submodules, without cloning, and without downloading anything by hand.

A repository publishes by committing a `crossrepo.yml` in the `results/`
directory at its root, naming the files it wants others to see and saying what
each one holds. Everything else in `results/` stays private to that project.

Repositories are read wherever they are: clones on this machine, clones on a
server reached over ssh, and repositories on GitHub read over the API without
being cloned at all.

In [1]:
import vscodenb
import crossrepo
import pandas as pd

### `crossrepo.yml`

To register files or directories as data available to `crossrepo`, add a `crossrepo.yml` to the repository `results`. Keys are file names, paths or globs and values say what the file holds. You can list parquet folders as if they were files, since they load as such.

```yml
files:
  hits.csv: Genome-wide association hits, p < 5e-8
  qc/summary.csv: Per-sample genotyping quality
  by_chrom: Per-chromosome effect sizes, one file per chromosome
```

A key beginning with `/` is a path from the repository root rather than from the
`crossrepo.yml`, which publishes a file that is kept elsewhere in the repository.
Any tracked file can be named this way; `..` cannot be used to climb out.

```yml
files:
  hits.csv: In the results directory, as usual
  /data/reference/samples.csv: Somewhere else in the repository
  /data/raw/*.tsv: A pattern, matched against the path from the root
```


## Config

`crossrepo` configuration is read from a config file that can be generated with `crossrepo config --init` and shown using:

In [2]:
crossrepo.active_config()

Config(roots=[],
       owners=['munch-group'],
       repos=[],
       asset_dirs=['results', 'data'],
       include=['*.csv',
                '*.tsv',
                '*.txt',
                '*.parquet',
                '*.pq',
                '*.h5',
                '*.hdf',
                '*.hdf5',
                '*.store',
                '*.json',
                '*.jsonl',
                '*.xlsx',
                '*.bed',
                '*.gff',
                '*.vcf',
                '*.vcf.gz',
                '*.pkl',
                '*.pickle',
                '*.npy',
                '*.npz',
                '*.feather',
                '*.zarr'],
       exclude=['*.png',
                '*.pdf',
                '*.svg',
                '*.html',
                '*.md',
                '.gitkeep',
                '*.log'],
       min_bytes=0,
       max_bytes=0)

You can also generate one on the fly in the notebook and pass along to each crossrepo function:

In [3]:
cfg = crossrepo.config.Config(
    repos=['munch-group/relate1Kgenomes', 'munch-group/atlas-variant-ages'], 
    include=['*.parquet', '*.csv', '*.tsv'],
    exclude=['.store', '*.fa', '*.vcf', '*.bim', '*.fam', '*.ped', '*.bam'],
)
crossrepo.use_config(cfg)
crossrepo.active_config() 

Config(roots=[],
       owners=[],
       repos=['munch-group/relate1Kgenomes', 'munch-group/atlas-variant-ages'],
       asset_dirs=['results'],
       include=['*.parquet', '*.csv', '*.tsv'],
       exclude=['.store', '*.fa', '*.vcf', '*.bim', '*.fam', '*.ped', '*.bam'],
       min_bytes=0,
       max_bytes=0)

In [4]:
crossrepo.use_config(None)  # back to ~/.config/crossrepo/config.toml

Config(roots=[],
       owners=['munch-group'],
       repos=[],
       asset_dirs=['results', 'data'],
       include=['*.csv',
                '*.tsv',
                '*.txt',
                '*.parquet',
                '*.pq',
                '*.h5',
                '*.hdf',
                '*.hdf5',
                '*.store',
                '*.json',
                '*.jsonl',
                '*.xlsx',
                '*.bed',
                '*.gff',
                '*.vcf',
                '*.vcf.gz',
                '*.pkl',
                '*.pickle',
                '*.npy',
                '*.npz',
                '*.feather',
                '*.zarr'],
       exclude=['*.png',
                '*.pdf',
                '*.svg',
                '*.html',
                '*.md',
                '.gitkeep',
                '*.log'],
       min_bytes=0,
       max_bytes=0)

### GitHub remote repositories

Search all github repos under these accounts:


In [5]:
cfg = crossrepo.config.Config(
    owners=["munch-group", ], 
)
crossrepo.use_config(cfg)
crossrepo.refresh()

reading GitHub:   0%|          | 0/157 [00:00<?, ?repo/s]

,owner,repo,name,description,get,date,github,path,dir,bytes,tags,lfs
0,munch-group,atlas-variant-ages,atlas_variant_ages.parquet,Variant ages using Abers method,github,2026-09-06,munch-group/atlas-variant-ages,results/atlas_variant_ages.parquet,results,1890901982,,False
1,munch-group,crossrepo,dummy.txt,Dummy text file,github,2026-09-11,munch-group/crossrepo,results/dummy.txt,results,12,,False
2,munch-group,hic-xy-sperm,all_genes.h5,all genes,github,2026-09-06,munch-group/hic-xy-sperm,results/all_genes.h5,results,2873280,,False
3,munch-group,hic-xy-sperm,segments_100000.csv,100kb segments,github,2026-09-06,munch-group/hic-xy-sperm,results/segments_100000.csv,results,1681136,,False
4,munch-group,hic-xy-sperm,segments_50000.csv,50kb segments,github,2026-09-06,munch-group/hic-xy-sperm,results/segments_50000.csv,results,3370795,,False
5,munch-group,hic-xy-sperm,segments_500000.csv,500kb segments,github,2026-09-06,munch-group/hic-xy-sperm,results/segments_500000.csv,results,334832,,False
6,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,Genome SNPs (hg38) with a log p-value below -2w,github,2026-09-06,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,results,82477487,,False


Once `refresh` has generated the cache, it can be listed with `list`:

In [6]:
crossrepo.list()

,owner,repo,name,description,get,date,github,path,dir,bytes,tags,lfs
0,munch-group,atlas-variant-ages,atlas_variant_ages.parquet,Variant ages using Abers method,github,2026-09-06,munch-group/atlas-variant-ages,results/atlas_variant_ages.parquet,results,1890901982,,False
1,munch-group,crossrepo,dummy.txt,Dummy text file,github,2026-09-11,munch-group/crossrepo,results/dummy.txt,results,12,,False
2,munch-group,hic-xy-sperm,all_genes.h5,all genes,github,2026-09-06,munch-group/hic-xy-sperm,results/all_genes.h5,results,2873280,,False
3,munch-group,hic-xy-sperm,segments_100000.csv,100kb segments,github,2026-09-06,munch-group/hic-xy-sperm,results/segments_100000.csv,results,1681136,,False
4,munch-group,hic-xy-sperm,segments_50000.csv,50kb segments,github,2026-09-06,munch-group/hic-xy-sperm,results/segments_50000.csv,results,3370795,,False
5,munch-group,hic-xy-sperm,segments_500000.csv,500kb segments,github,2026-09-06,munch-group/hic-xy-sperm,results/segments_500000.csv,results,334832,,False
6,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,Genome SNPs (hg38) with a log p-value below -2w,github,2026-09-06,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,results,82477487,,False


### Local working copies

You can use the `roots` key to specify folders where you have checked out working copies of repositories. When files exist on the same filesystem, `crossrepo.get()` returns a **hard link** rather than a copy to not take up extra disk space.

In [10]:
cfg = crossrepo.config.Config(
    # owners=["munch-group", ], 
    roots = ["~/", ]
)
crossrepo.use_config(cfg)
crossrepo.refresh()

scanning clones:   0%|          | 0/23 [00:00<?, ?repo/s]

,owner,repo,name,description,date,github,path,dir,bytes,tags,lfs
0,munch-group,crossrepo,dummy.csv,Dome dummy csv file sym linked from steps,2026-09-11,munch-group/crossrepo,results/dummy.csv,results,0,,False
1,munch-group,crossrepo,dummy.txt,Dummy text file,2026-09-11,munch-group/crossrepo,results/dummy.txt,results,12,,False
2,munch-group,crossrepo,tester.parquet,Tester symlink to dir,2026-09-11,munch-group/crossrepo,results/tester.parquet,results,0,,False


Working copies of repositories takes precedence over remotes found on github.  

### Remote working copies

You can also access working copies of repositories on remote servers, like GenomeDK, if you have set up password-less access using ssh keys. To make two-factor authentication with the genome.dk cluster more smooth, you can add this to your `~/.ssh/config` (replacing "kmt" with your own username on the cluster):

```txt
Host *
  ServerAliveInterval 60

Host gdk
    HostName        login.genome.au.dk
    User            kmt
    ControlMaster   auto
    ControlPath     ~/.ssh/cm-%r@%h:%p
    ControlPersist  4h
```

and then use the `gdk` alias when specifying a root in configs:

In [ ]:
cfg = crossrepo.config.Config(
    # owners=["munch-group", ], 
    roots = ["gdk:xy-drive/people/kmt", "gdk:primatediversity/people/kmt", "~/", ]
)
crossrepo.use_config(cfg)
crossrepo.refresh()

scanning clones:   0%|          | 0/42 [00:00<?, ?repo/s]

,owner,repo,name,get,description,date,github,path,dir,bytes,tags,lfs
0,munch-group,atlas-variant-ages,atlas_variant_ages.parquet,gdk,Variant ages using Abers method,2026-09-06,munch-group/atlas-variant-ages,results/atlas_variant_ages.parquet,results,1890901982,,False
1,munch-group,crossrepo,dummy.csv,local,Dome dummy csv file sym linked from steps,2026-09-11,munch-group/crossrepo,results/dummy.csv,results,0,,False
2,munch-group,crossrepo,dummy.txt,local,Dummy text file,2026-09-11,munch-group/crossrepo,results/dummy.txt,results,12,,False
3,munch-group,crossrepo,tester.parquet,local,Tester symlink to dir,2026-09-11,munch-group/crossrepo,results/tester.parquet,results,0,,False
4,munch-group,hic-xy-sperm,all_genes.h5,gdk,all genes,2026-09-06,munch-group/hic-xy-sperm,results/all_genes.h5,results,2873280,,False
5,munch-group,hic-xy-sperm,segments_100000.csv,gdk,100kb segments,2026-09-06,munch-group/hic-xy-sperm,results/segments_100000.csv,results,1681136,,False
6,munch-group,hic-xy-sperm,segments_50000.csv,gdk,50kb segments,2026-09-06,munch-group/hic-xy-sperm,results/segments_50000.csv,results,3370795,,False
7,munch-group,hic-xy-sperm,segments_500000.csv,gdk,500kb segments,2026-09-06,munch-group/hic-xy-sperm,results/segments_500000.csv,results,334832,,False
8,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,gdk,Genome SNPs (hg38) with a log p-value below -2w,2026-09-06,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,results,82477487,,False
9,munch-group,vep-data,vep.parquet,gdk,Parquet formatted VEP info from ensembl,2026-09-11,munch-group/vep-data,results/vep.parquet,results,31840511289,,False


Accessing working copies is useful if they contain temporary result files too big for github commit. In that case, you add a symlink (E.g. `dummy.csv -> ../steps/out/dummy.csv`) to your results folder and then add the symlink to `crossrepo.yml`:

```yml
files:
  dummy.csv: Dummy csv file sym linked from steps
```

However, since `git` only tracks the relative path of the symlink and not the file it links to, you need to generate a version identifier for the linked file. You do that by running `crossrepo stamp` in the repository. This adds a version hash to `crossrepo.yml`. Once you have committed the updated `crossrepo.yml`, the large file is available through `crossrepo`.

```yml
files:
  dummy.csv:
    description: Dummy csv file sym linked from steps
    sha256: "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
    size: 0
```

## Repos

In [3]:
crossrepo.repos()

KeyboardInterrupt: 

## List

`crossrepo.list()` is the python side of `crossrepo list`.

In [ ]:
crossrepo.list()

,owner,repo,name,description,date,github,path,dir,bytes,tags,lfs
0,munch-group,CTCF-inversion-data,ctcf_encode_files.tsv,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_encode_files.tsv,results,115265,,False
1,munch-group,CTCF-inversion-data,ctcf_sites_hg38.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38.parquet,results,11741434,,False
2,munch-group,CTCF-inversion-data,ctcf_sites_hg38_chm13.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38_chm13.parquet,results,17454688,,False
3,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,Genome SNPs (hg38) with a log p-value below -2w,2026-09-02,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,results,82477487,,False
4,munch-group,tree-stats,dummy.csv,Some dummy csv file,2026-08-30,munch-group/tree-stats,results/dummy.csv,results,19,,False


In [ ]:
crossrepo.list(brief=True)

,owner,repo,name,size,description,date
0,munch-group,CTCF-inversion-data,ctcf_encode_files.tsv,0.1 MB,blah,2026-09-02
1,munch-group,CTCF-inversion-data,ctcf_sites_hg38.parquet,11.7 MB,blah,2026-09-02
2,munch-group,CTCF-inversion-data,ctcf_sites_hg38_chm13.parquet,17.5 MB,blah,2026-09-02
3,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,82.5 MB,Genome SNPs (hg38) with a log p-value below -2w,2026-09-02
4,munch-group,tree-stats,dummy.csv,0.0 MB,Some dummy csv file,2026-08-30


In [ ]:
crossrepo.list(version=True)

,owner,repo,name,description,date,github,path,dir,bytes,tags,lfs,version
0,munch-group,CTCF-inversion-data,ctcf_encode_files.tsv,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_encode_files.tsv,results,115265,,False,14ebf0da170dfe4c0baca2840ff8d3be81640b9c
1,munch-group,CTCF-inversion-data,ctcf_sites_hg38.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38.parquet,results,11741434,,False,14ebf0da170dfe4c0baca2840ff8d3be81640b9c
2,munch-group,CTCF-inversion-data,ctcf_sites_hg38_chm13.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38_chm13.parquet,results,17454688,,False,14ebf0da170dfe4c0baca2840ff8d3be81640b9c
3,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,Genome SNPs (hg38) with a log p-value below -2w,2026-09-02,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,results,82477487,,False,590c4f17d5a2ff765e61b6df9a1fa88318c9760d
4,munch-group,tree-stats,dummy.csv,Some dummy csv file,2026-08-30,munch-group/tree-stats,results/dummy.csv,results,19,,False,770170789d73428efb0193d4cf04a353587db9cd


## Get

In [33]:
tmp_path = crossrepo.get("atlas-variant-ages", "atlas_variant_ages.parquet")

Add version="619b60e3af78bed6494222f82f754a3863249fe9" to pin this version.


In [9]:
df = pd.read_parquet(tmp_path)
df.head()

,chrom,pop,pos,p_half_freq,p_two_alleles
0,chr1,ACB,817341,-0.473037,-2.88930
1,chr1,ACB,892733,-1.231820,-2.30420
2,chr1,ACB,893360,-1.231820,-2.30420
3,chr1,ACB,897538,-1.319140,-4.10702
4,chr1,ACB,901516,-1.319140,-3.19570


In [10]:
filters = [
    ('pos', '>=', 892733), 
    ('pos', '<', 901516),
    ('pop', '==', "ACB"),
    ('chrom', '==', "chrX"),
]
pd.read_parquet(tmp_path, filters=filters)

,chrom,pop,pos,p_half_freq,p_two_alleles
0,chrX,ACB,893285,-1.273310,-4.51818
1,chrX,ACB,895075,-1.713310,-4.48943
2,chrX,ACB,895885,-1.274320,-2.17077
3,chrX,ACB,896099,-1.274320,-2.17077
4,chrX,ACB,896561,-1.713310,-4.48943
5,chrX,ACB,897491,-1.355870,-5.26528
6,chrX,ACB,897556,-0.750727,-3.91539
7,chrX,ACB,897658,-0.574665,-2.00685
8,chrX,ACB,898415,-0.301048,-4.08054
9,chrX,ACB,899070,-0.574665,-2.00685


In [ ]:
tmp_path = crossrepo.get("munch-group/relate1Kgenomes", "results/relate_snps_p_vals.parquet")

Add version="590c4f17d5a2ff765e61b6df9a1fa88318c9760d" to pin this version.


In [ ]:
df = pd.read_parquet(tmp_path)
df.head()

,variant_id,chrom,pos,ref,alt,anc,age_mode,age_lo95ci,age_hi95ci
0,rs537182016,1,10539,C,A,.,527.489,17.852,1118.33
1,rs558604819,1,10642,G,A,.,11418.100,10038.000,12847.50
2,rs575272151,1,11008,C,G,.,16174.800,14789.800,17615.10
3,rs544419019,1,11012,C,G,.,20599.300,18875.200,22362.90
4,rs561109771,1,11063,T,G,.,5802.930,4794.360,6848.18


In [16]:
# silent, because this is still the current version
tmp_path = crossrepo.get("munch-group/relate1Kgenomes", 
                       "results/relate_snps_p_vals.parquet",
                       version="590c4f17d5a2ff765e61b6df9a1fa88318c9760d")

The file name is enough when it is unambiguous; otherwise give as much of the
path as it takes.

In [17]:
crossrepo.get("relate1Kgenomes", "relate_snps_p_vals.parquet")

Add version="590c4f17d5a2ff765e61b6df9a1fa88318c9760d" to pin this version.


PosixPath('/Users/kmt/.cache/crossrepo/files/munch-group__relate1Kgenomes/590c4f17d5a2ff765e61b6df9a1fa88318c9760d/results/relate_snps_p_vals.parquet')

## History

The catalog stamps every file with the repository's current commit.
`crossrepo.versions()` shows the commits in which the file itself changed, which
is the useful set to pin.

In [18]:
crossrepo.versions("relate1Kgenomes", "relate_snps_p_vals.parquet")

,version,date,bytes,parts,tags,subject
0,f4f1fc621cb9dfac3610f7f47d90caf640b43571,2026-09-01,82477487,3,,Added parquet fixed data


## When nothing shows up

An empty catalog has several ordinary causes and they look alike from outside.
`crossrepo.diagnose()` says which it is, reading only local git.

In [14]:
print("\n".join(crossrepo.diagnose(crossrepo.Config(roots=["~/no-such-place"]))))

  ~/no-such-place: no such directory
  github: no owners or repos configured, so nothing is read from GitHub


In [ ]:
import sys
from crossrepo import location
print("interactive:", location._interactive())
print("kernel     :", location._kernel())
print("bridged    :", location._bridged())
print("stdin      :", type(sys.stdin), getattr(sys.stdin, "isatty", lambda: "n/a")())
print("options    :", location.ssh_options("kmt@login.genome.au.dk"))
